# Part 2 — Feature Engineering & Data Preparation

## Feature Engineering

In [13]:
df_model = df.copy()

# Drop the Week column we created during EDA â€” will recreate cleanly
if "Week" in df_model.columns:
    df_model = df_model.drop(columns=["Week"])

# Time-based features
df_model["Year"]      = df_model["Date"].dt.year
df_model["Month"]     = df_model["Date"].dt.month
df_model["Week"]      = df_model["Date"].dt.strftime('%U').astype(int)
df_model["Quarter"]   = df_model["Date"].dt.quarter
df_model["DayOfYear"] = df_model["Date"].dt.dayofyear

# Holiday flags
df_model["IsBlackFriday"]  = ((df_model["Month"] == 11) & (df_model["Week"] >= 47)).astype(int)
df_model["IsChristmas"]    = ((df_model["Month"] == 12) & (df_model["Week"] >= 51)).astype(int)
df_model["IsThanksgiving"] = ((df_model["Month"] == 11) & (df_model["Week"] >= 46) & (df_model["Week"] < 47)).astype(int)

# Store type encoding
df_model["Type_A"] = (df_model["Type"] == "A").astype(int)
df_model["Type_B"] = (df_model["Type"] == "B").astype(int)
df_model["Type_C"] = (df_model["Type"] == "C").astype(int)

# Lag features (requires sorting by store/dept/date)
df_model = df_model.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
for lag in [1, 4, 8, 52]:
    df_model[f"Sales_Lag_{lag}"] = df_model.groupby(["Store", "Dept"])["Weekly_Sales"].shift(lag)

# Rolling statistics
for window in [4, 8, 12]:
    df_model[f"Sales_MA_{window}"]  = df_model.groupby(["Store", "Dept"])["Weekly_Sales"].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean()
    )
    df_model[f"Sales_STD_{window}"] = df_model.groupby(["Store", "Dept"])["Weekly_Sales"].transform(
        lambda x: x.rolling(window=window, min_periods=1).std()
    )

print(f"Features before: {len(df.columns)}")
print(f"Features after : {len(df_model.columns)}")

Features before: 12
Features after : 32


## Data Preparation & Train/Val Split

In [15]:
# Drop rows with NaN caused by lag features (first ~52 weeks per Store/Dept)
df_model_clean = df_model.dropna()
print(f"Rows before cleaning : {len(df_model):,}")
print(f"Rows after  cleaning : {len(df_model_clean):,}")
print(f"Rows removed (NaN lags): {len(df_model) - len(df_model_clean):,}")

available_features = [
    "Store", "Dept", "Size",
    "Week", "Month", "Quarter", "DayOfYear",
    "IsHoliday", "IsBlackFriday", "IsChristmas", "IsThanksgiving",
    "Type_A", "Type_B", "Type_C",
    "Sales_Lag_1", "Sales_Lag_4", "Sales_Lag_8", "Sales_Lag_52",
    "Sales_MA_4",  "Sales_MA_8",  "Sales_MA_12",
    "Sales_STD_4", "Sales_STD_8", "Sales_STD_12",
]

# Time-based split (no data leakage)
split_date = "2012-08-01"
train = df_model_clean[df_model_clean["Date"] <  split_date]
val   = df_model_clean[df_model_clean["Date"] >= split_date]

X_train, y_train = train[available_features], train["Weekly_Sales"]
X_val,   y_val   = val[available_features],   val["Weekly_Sales"]

print(f"\nTraining  : {len(train):,} samples  ({train['Date'].min().date()} â†’ {train['Date'].max().date()})")
print(f"Validation: {len(val):,}  samples  ({val['Date'].min().date()}   â†’ {val['Date'].max().date()})")
print(f"Features  : {len(available_features)}")

Rows before cleaning : 421,570
Rows after  cleaning : 261,083
Rows removed (NaN lags): 160,487

Training  : 223,218 samples  (2011-02-04 â†’ 2012-07-27)
Validation: 37,865  samples  (2012-08-03   â†’ 2012-10-26)
Features  : 24
